
# Stage 4: Exploratory Data Analysis and Feature Engineering

## 4.1 Introduction

The purpose of this stage is to explore hotel booking patterns and identify factors related to cancellations. We will use statistics and visualizations to understand the data and create useful features for machine learning.


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("hotel_bookings_cleaned.csv")

print("Dataset Shape:", df.shape)
display(df.head())



## 4.2 Hotel Booking Cancellation Rate

We will calculate the number and percentage of canceled and non-canceled bookings. This helps us understand the overall cancellation pattern in the dataset.


In [1]:
print("Cancellation Counts:")
print(df["is_canceled"].value_counts())

print("\nCancellation Percentages:")
print(df["is_canceled"].value_counts(normalize=True) * 100)

sns.countplot(data=df, x="is_canceled")
plt.title("Hotel Booking Cancellation Distribution")
plt.xlabel("Cancellation Status (0 = Not Canceled, 1 = Canceled)")
plt.ylabel("Number of Bookings")
plt.show()


Cancellation Counts:


NameError: name 'df' is not defined


### Interpretation

The results show that 62.92% of hotel bookings were not canceled, while 37.08% were canceled. This indicates that cancellations represent a significant proportion of bookings, highlighting the importance of predicting cancellations to support hotel planning and reduce potential revenue losses.



## 4.3 Cancellation Rate by Hotel Type

This analysis compares cancellation rates between City Hotel and Resort Hotel to identify differences in booking cancellation patterns.


In [ ]:

hotel_cancel_rate = df.groupby("hotel")["is_canceled"].mean() * 100

print("Cancellation Rate by Hotel Type:")
print(hotel_cancel_rate)

sns.barplot(data=df, x="hotel", y="is_canceled", errorbar=None)
plt.title("Cancellation Rate by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("Cancellation Rate")
plt.show()



### Interpretation

City Hotels have a higher cancellation rate (41.79%) compared to Resort Hotels (27.77%). This suggests that hotel type is associated with cancellation behavior and may be a useful feature for predicting cancellations.



## 4.4 Lead Time and Cancellations

This analysis examines the relationship between booking lead time and cancellation status. We will compare the average lead time of canceled and non-canceled bookings.


In [ ]:

lead_time_avg = df.groupby("is_canceled")["lead_time"].mean()

print("Average Lead Time by Cancellation Status:")
print(lead_time_avg)

sns.barplot(data=df, x="is_canceled", y="lead_time", errorbar=None)
plt.title("Average Lead Time by Cancellation Status")
plt.xlabel("Cancellation Status (0 = Not Canceled, 1 = Canceled)")
plt.ylabel("Average Lead Time (Days)")
plt.show()



### Interpretation

Canceled bookings have a higher average lead time (144.89 days) compared to non-canceled bookings (80.08 days). This suggests that bookings made further in advance are associated with a higher likelihood of cancellation. Lead time may therefore be an important feature for predicting cancellations.



## 4.5 Room Prices and Cancellations

This analysis compares the average daily rate (ADR) between canceled and non-canceled bookings to investigate whether room prices are associated with cancellation behavior.


In [ ]:

adr_avg = df.groupby("is_canceled")["adr"].mean()

print("Average Daily Rate by Cancellation Status:")
print(adr_avg)

sns.barplot(data=df, x="is_canceled", y="adr", errorbar=None)
plt.title("Average Daily Rate by Cancellation Status")
plt.xlabel("Cancellation Status (0 = Not Canceled, 1 = Canceled)")
plt.ylabel("Average Daily Rate (ADR)")
plt.show()



### Interpretation

Canceled bookings have a slightly higher average daily rate (105.02) compared to non-canceled bookings (100.17). The difference is relatively small, suggesting that room price alone may not be a strong indicator of cancellation behavior.



## 4.6 Deposit Type and Cancellations

This analysis compares cancellation rates across different deposit types to understand whether deposit policies are associated with booking cancellation behavior.


In [ ]:

deposit_cancel_rate = df.groupby("deposit_type")["is_canceled"].mean() * 100

print("Cancellation Rate by Deposit Type:")
print(deposit_cancel_rate.round(2))

sns.barplot(data=df, x="deposit_type", y="is_canceled", errorbar=None)
plt.title("Cancellation Rate by Deposit Type")
plt.xlabel("Deposit Type")
plt.ylabel("Cancellation Rate")
plt.show()



### Interpretation

Non-refundable bookings have the highest cancellation rate (99.36%), compared to No Deposit bookings (28.40%) and Refundable bookings (22.22%). This suggests that deposit type is strongly associated with cancellation behavior and may be an important feature for predicting cancellations. However, this does not mean that non-refundable deposits cause cancellations.



## 4.7 Customer Type and Cancellations

This analysis compares cancellation rates across customer types to understand whether customer categories are associated with booking cancellation behavior.


In [ ]:

customer_summary = df.groupby("customer_type")["is_canceled"].agg(
    bookings="count",
    cancellation_rate="mean"
)

customer_summary["cancellation_rate"] *= 100

print("Cancellation Rate by Customer Type:")
print(customer_summary.round(2))

sns.barplot(
    data=df,
    x="customer_type",
    y="is_canceled",
    errorbar=None
)

plt.title("Cancellation Rate by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Cancellation Rate")
plt.xticks(rotation=20)
plt.show()



### Interpretation

Transient customers have the highest cancellation rate (40.79%), followed by Contract customers (30.99%) and Transient-Party customers (25.45%). Group customers have the lowest cancellation rate (10.10%). This suggests that customer type is associated with cancellation behavior and may be useful for predicting cancellations.



## 4.8 Monthly Cancellation Patterns

This analysis compares cancellation rates across arrival months to identify seasonal patterns in hotel booking cancellations.


In [ ]:

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_summary = df.groupby("arrival_date_month")["is_canceled"].agg(
    bookings="count",
    cancellation_rate="mean"
).reindex(month_order)

monthly_summary["cancellation_rate"] *= 100

print("Monthly Cancellation Rates:")
print(monthly_summary.round(2))

plt.figure(figsize=(12, 5))

sns.barplot(
    data=df,
    x="arrival_date_month",
    y="is_canceled",
    order=month_order,
    errorbar=None
)

plt.title("Cancellation Rate by Arrival Month")
plt.xlabel("Arrival Month")
plt.ylabel("Cancellation Rate")
plt.xticks(rotation=45)
plt.show()



### Interpretation

Cancellation rates vary across arrival months. June has the highest cancellation rate (41.49%), followed by April (40.78%), while January has the lowest rate (30.50%). This suggests that arrival month may be associated with cancellation behavior and could be useful for predicting cancellations.



## 4.9 Feature Engineering

In this step, we create new features from existing booking information. These features summarize the number of guests, length of stay, and previous booking history. They may help improve the performance of machine learning models.


In [ ]:

df["total_guests"] = (
    df["adults"] + df["children"] + df["babies"]
)

df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["total_previous_bookings"] = (
    df["previous_cancellations"] +
    df["previous_bookings_not_canceled"]
)

print("Dataset Shape After Feature Engineering:", df.shape)

display(df[[
    "total_guests",
    "total_nights",
    "total_previous_bookings"
]].head())



### Interpretation

Three new features were created: total_guests, total_nights, and total_previous_bookings. These features summarize the number of guests, length of stay, and previous booking history. The dataset now contains 36 columns, and these features may help improve cancellation prediction.



## 4.10 Preparing the Dataset for Machine Learning

In this step, we remove columns that could cause data leakage or are unsuitable for direct model training. We keep the cancellation target and the relevant booking features for Stage 5.


In [ ]:

columns_to_drop = [
    "reservation_status",
    "reservation_status_date",
    "arrival_date"
]

df_model = df.drop(columns=columns_to_drop)

print("Dataset Shape for Modeling:", df_model.shape)

print("\nMissing Values:", df_model.isnull().sum().sum())

print("\nTarget Variable:")
print(df_model["is_canceled"].value_counts())

display(df_model.head())



### Interpretation

The dataset was prepared for machine learning by removing reservation_status, reservation_status_date, and arrival_date. The resulting dataset contains 119,209 rows and 33 columns, with no missing values. The target variable, is_canceled, was retained for predicting hotel booking cancellations.


In [ ]:

df_model.to_csv("hotel_bookings_model_ready.csv", index=False)

print("Model-ready dataset saved successfully!")
print("Final Dataset Shape:", df_model.shape)
